# verl.workers

- verl.workers 模块是实现其核心计算任务的执行单元。
- 你可以将 Worker 理解为在分布式环境中扮演特定角色的“演员”，它们负责执行具体的计算，如模型推理、训练更新、价值评估等。
- VeRL 的架构设计将控制流（做什么）与计算流（怎么做）分离，而 Worker 正是计算流的具体承载者。


verl.workers 模块是 VeRL 框架的核心执行单元，它通过一套高度模块化和抽象的设计，实现了对不同训练和推理后端（如 FSDP、Megatron-LM、vLLM、SGLang）的统一支持。

为了统一多样的后端，verl.workers 引入了一个关键的抽象层——引擎（Engine）


1. BaseEngine 抽象接口

这是一个抽象基类，定义了模型训练和推理的通用操作契约，如 initialize()、train_mode()、forward_backward_batch()、optimizer_step() 等。上层的工作者（Worker）只与这个统一的接口交互，而无需关心底层是 FSDP 还是 Megatron。

2. 具体引擎实现

BaseEngine 派生出针对不同后端的实现，如 FSDPEngine 和 MegatronEngine。这些具体的引擎类负责封装和实现对应分布式框架的复杂细节。

3. 统一的工作者

得益于引擎抽象，VeRL 不再需要为每个后端维护一套独立的 Worker 代码（如旧的 fsdp_worker 和 megatron_worker）。现在，像 ActorRolloutRefWorker 和 TrainingWorker 这样的通用工作者，通过持有不同的 Engine 实例，就能灵活地支持多种后端，极大地简化了代码维护和新后端的接入。



## 主要工作者（Worker）类型

    verl.workers 根据强化学习训练流程中的不同职责，定义了多种工作者：
    
- ActorRolloutRefWorker: 这是最核心的工作者，用于强化学习任务。它集成了策略网络（Actor）的训练、利用推理后端进行文本生成（Rollout）以及参考模型（Reference Model）的功能。
- TrainingWorker: 一个更通用的训练工作者，包含基础的训练能力，可直接用于监督微调（SFT）等任务。
- CriticWorker: 负责价值网络（Critic）的训练，用于估计状态价值，为策略梯度计算提供基线。
- RewardModelWorker: 用于加载和运行奖励模型，为生成的序列提供奖励分数。

### 🧩 核心 Worker 角色
1. Actor Worker (verl.workers.actor)

    这是 RL 训练中的核心角色，负责策略（Policy）的执行和更新。
- 功能：
    - 生成 (Rollout)：接收提示词（prompts），利用当前策略模型生成文本序列。
    - 训练 (Update)：根据计算出的优势（Advantage）等信号，更新自身的模型参数。
- 实现：为了支持不同的并行策略，Actor Worker 有多种实现，如 DataParallelActor 和 MegatronActor

2. Critic Worker (verl.workers.critic)

    Critic 负责评估 Actor 生成结果的价值。
- 功能：
    - 价值估计：接收 Actor 生成的序列，预测其累积奖励（Value）。
    - 训练 (Update)：通过比较预测价值和实际回报（Return），更新自身的价值网络（Value Network）参数，以更准确地评估状态。
- 实现：同样，Critic 也有 DataParallelCritic 和 MegatronCritic 等实现。

3. Reward Model Worker (verl.workers.reward_model)

这个 Worker 专门用于提供奖励信号。
- 功能：对 Actor 生成的文本序列进行打分，这个分数是 RL 算法（如 PPO）优化策略的关键依据。
- 实现：通常是一个独立的模型，其实现也支持分布式部署。

4. Rollout Worker (verl.workers.rollout)

Rollout Worker 专注于高效的文本生成。
- 功能：为了追求极致的推理吞吐，VeRL 允许将生成任务卸载到专门的推理引擎上。Rollout Worker 就是这些引擎的封装。
- 实现：例如，vllm_rollout 模块就是对高性能推理库 vLLM 的封装，它利用 vLLM 的连续批处理和 PagedAttention 等技术来加速生成过程。

## ⚙️ 管理与调度机制

VeRL 通过一套精巧的机制来管理和调度这些 Worker，使其能够高效协同。

1. <font color='red'>基于 Ray 的分布式执行</font>

VeRL 构建在 Ray 之上，每个 Worker（如 Actor、Critic）在启动时都会被实例化为一个 Ray Actor。
- 资源隔离：通过 Ray 的 PlacementGroup，可以精确地将不同的 Worker 分配到指定的 GPU 上。例如，你可以将计算密集的 Actor 放在高性能 GPU 集群上，而将 Critic 放在另一组 GPU 上。
- 独立运行：每个 Worker 作为独立的 Ray Actor 运行，拥有自己的内存空间和执行线程，通过远程过程调用（RPC）进行通信，避免了单点瓶颈。

2. <font color='red'>分片管理器 (verl.workers.sharding_manager)</font>

这是 VeRL 实现高性能的关键组件之一，它解决了 RL 训练中模型在不同阶段需要不同并行策略的难题。
- 问题：Actor 在“生成”阶段适合使用张量并行（TP），而在“训练”阶段则更适合数据并行（DP）或 FSDP。传统方法在切换时需要保存和加载整个模型状态，开销巨大。
- 解决方案：sharding_manager 负责管理模型参数在不同并行模式（如从 TP 切换到 FSDP）之间的动态重分片（Re-sharding）。它通过预规划和内存复用技术，极大地减少了参数重新分布的通信开销和显存占用，实现了“零冗余”内存管理。

3. <font color='red'>数据传输协议 (verl.workers 内部)</font>

为了协调不同 Worker 之间的数据流动，VeRL 内部设计了一套数据传输协议。

- 分发 (Dispatch)：将数据（如一个批次的提示词）按照预设的并行策略（如按 DP 维度切分）分发给各个 Worker。
- 收集 (Collect)：将各个 Worker 的计算结果（如生成的序列）收集并聚合起来，传递给下一个阶段的 Worker。


## 🔗 与后端框架的集成

verl.workers 通过专门的模块与主流的训练和推理框架深度集成。

### 训练后端

通过 FSDPEngine 和 MegatronEngine，verl.workers 支持了两种主流的分布式训练方案：

| 训练后端 | 支持方式 | 典型场景 |
| :--- | :--- | :--- |
| PyTorch FSDP | `FSDPEngine` | 训练 13B 等中等规模模型，或与 Hugging Face 生态紧密集成。 |
| Megatron-LM | `MegatronEngine` | 训练 70B+ 的超大模型或 MoE 模型，利用其张量并行（TP）、流水线并行（PP）和专家并行（EP）能力。 |







verl.workers.fsdp_workers 和 verl.workers.megatron_workers 是 VeRL 框架中实现具体计算任务的两大核心后端模块。<b>它们为强化学习中的各个角色（如 Actor、Critic）提供了基于不同分布式训练策略的具体实现。</b>

你可以将这两个模块理解为 VeRL 框架的两种“引擎”选项：
- fsdp_workers：基于 PyTorch 原生的 FSDP (Fully Sharded Data Parallel)，如同一个“自动挡”引擎，易于使用，兼容性好，适合快速迭代和中小规模模型。
- megatron_workers：基于 NVIDIA 的 Megatron-LM 框架，如同一个“手动赛车”引擎，配置复杂但性能极致，专为训练超大规模模型而生。

VeRL 通过统一的 Worker 接口（如 ActorRolloutRefWorker）<b>将这两种实现封装起来</b>，使得上层训练逻辑可以无缝切换后端，而无需关心底层细节。

#### ⚙️ FSDP Workers (verl.workers.fsdp_workers)
这个模块为 VeRL 提供了基于 PyTorch FSDP 的分布式实现。<font color='red'>FSDP 是一种动态分片策略，它将模型的参数、梯度和优化器状态切分到不同的 GPU 上，在计算时才进行必要的通信和聚合。</font>

<b>核心特性</b>

1. 易于使用与高兼容性
- 接入简单：FSDP 是 PyTorch 的原生功能，因此 fsdp_workers 可以非常方便地加载和运行任何 Hugging Face 格式的模型，无需进行复杂的模型结构修改。
- 开发友好：对于学术界和算法原型验证来说，FSDP 后端足够高效且调试方便，是快速启动 RLHF 实验的理想选择。
2. 主要并行策略
- 数据并行 (Data Parallelism, DP)：这是 FSDP 的基础，通过在不同 GPU 上处理不同的数据批次来提升吞吐量。
- 序列并行 (Sequence Parallelism, SP)：fsdp_workers 通过集成 Ulysses 算法，在注意力计算（Attention）的维度上实现了序列并行。这对于处理长文本序列至关重要，可以有效降低单卡的显存压力。
3. <b>关键 Worker 实现</b>
- <font color='red'>ActorRolloutRefWorker</font>: 这是 FSDP 后端中最核心的 Worker。<font color='green'>它巧妙地整合了 Actor（策略模型）、Rollout（生成）和 Reference（参考模型）三种角色。</font> <font color='red'>通过 FSDPVLLMShardingManager，它能在 FSDP 训练模式和 vLLM 推理模式之间高效地切换模型参数的分片方式。</font>
- <font color='red'>CriticWorker</font>: 负责价值网络的训练，其内部基于 DataParallelPPOCritic 实现，利用 FSDP 来计算价值损失并更新模型。
- <font color='red'>RewardModelWorker</font>: 用于加载和运行奖励模型，同样基于 FSDP 策略进行分布式推理。


#### 🚀 Megatron Workers (verl.workers.megatron_workers)

这个模块提供了基于 Megatron-LM 的分布式实现。Megatron-LM 是一个功能强大的框架，专为训练数千亿乃至万亿参数级别的模型而设计，提供了比 FSDP 更精细、更多维度的并行控制。

核心特性
1. 极致的并行效率
 - 5D 混合并行：这是 Megatron 的最大优势。megatron_workers 支持多种并行策略的组合，包括：
    - 张量并行 (TP)：将单个矩阵运算拆分到多个 GPU 上。
    - 流水线并行 (PP)：将模型的不同层分配到不同 GPU 上，形成流水线。
    - 专家并行 (EP)：专门用于高效训练 MoE (Mixture of Experts) 模型。
    - 上下文并行 (CP)：类似于 Ulysses，用于处理长序列。
    - 数据并行 (DP)：基础的数据批次并行。
 - <b>这种多维度并行能力使得 megatron_workers 能够高效地训练如 DeepSeek-V3 (671B) 等超大规模模型，这是 FSDP 难以企及的。</b>
 
2. 深度优化与定制
 - 性能优先：在工业界追求极致训练吞吐时，Megatron 后端是首选。它允许进行许多底层定制优化，以充分发挥硬件性能。
 - 全状态卸载：支持将参数、梯度和优化器状态完全卸载到 CPU 内存，进一步突破 GPU 显存的限制。
3. <b>关键 Worker 实现</b>
 - 该模块中的 Worker（如 ActorRolloutRefWorker, CriticWorker）在高层逻辑上与 FSDP 版本基本一致，但底层计算完全适配了 Megatron-LM 的 API 和并行原语。
 - 例如，<font color='red'>megatron_workers 中的 Actor 会利用 Megatron 的 3DHybridEngine 和 MegatronVLLMShardingManager 来实现训练与推理引擎间的高效权重转换，其切换速度远超传统方案。</font>
 
 
##### 有以下类
- MegatronWorker
- ActorRolloutRefWorker  核心类
- CriticWorker
- RewardModelWorker

######  🏗️ 核心类：ActorRolloutRefWorker

这是该模块中最关键的类，它将策略网络（Actor）、文本生成（Rollout）和参考网络（Reference）三种角色整合在一起。其代码逻辑主要围绕模型的初始化、参数同步和序列生成展开。

1. 模型初始化 (init_model)

init_model 方法负责在 GPU 上构建模型。与 FSDP 版本不同，它需要初始化 Megatron-LM 的分布式环境，并根据配置设置张量并行（TP）和流水线并行（PP）的维度。


In [ ]:
# 伪代码逻辑示意
def init_model(self):
    # 1. 初始化 Megatron 的分布式环境
    # 这会设置 up tensor parallel group, pipeline parallel group 等
    initialize_megatron(...)

    # 2. 根据配置构建 Actor 模型
    # 模型会被自动切分到不同的 TP 和 PP 组上
    self.actor_model = build_model(...)

    # 3. 同样地构建 Reference 模型
    self.ref_model = build_model(...)

2. 构建 Rollout 引擎 (_build_rollout)

这是 megatron_workers 与 FSDP 版本差异最大的地方。为了支持高效的文本生成，它同样会创建一个 vLLMRollout 实例，但参数的传递和同步方式完全不同。
 - FSDP 版本：vLLM 引擎独立加载模型权重，然后通过 FSDPVLLMShardingManager 在需要时从 FSDP 模型同步参数。
 - Megatron 版本：<font color='red'>由于模型是分布式存储在多个 GPU 上的，vLLM 引擎无法独立加载。因此，代码需要先从 Megatron 的分布式模型中收集所有参数，然后直接传递给 vLLM 引擎进行初始化。</font>

In [ ]:
# 伪代码逻辑示意，基于 verl/workers/megatron_workers.py 的 _build_rollout 方法
def _build_rollout(self):
    if self.config.rollout.name == 'vllm':
        from verl.workers.rollout.vllm_rollout import vLLMRollout
        
        # ⭐ 关键区别：从 Megatron 模型中获取参数
        # 1. 将模型参数加载到 CUDA
        self.hybrid_engine.load_params_to_cuda()
        # 2. 在所有 PP ranks 上 gather 参数
        self.hybrid_engine.allgather_params()
        # 3. 获取完整的参数列表
        params = self.hybrid_engine.get_all_params()
        
        # ⚠️ 注意：Megatron 后端目前仅支持 vLLM 的 'customized' 模式
        # 这意味着需要使用一个经过特殊修改的 vLLM 版本
        rollout = vLLMRollout(
            actor_module=params, # 直接传入参数，而不是模型路径
            config=self.config.rollout,
            tokenizer=self.tokenizer,
            model_hf_config=self.actor_model_config,
            train_tp=mpu.get_tensor_model_parallel_world_size() # 传入训练时的 TP 大小
        )
        return rollout

3. 序列生成 (generate_sequences)
该方法负责协调整个生成流程。它的核心是调用 Rollout 引擎，但在调用前后，需要处理复杂的参数同步和数据预处理。

In [ ]:
# 伪代码逻辑示意，基于 verl/workers/megatron_workers.py 的 generate_sequences 方法
@register(dispatch_mode=Dispatch.DP_COMPUTE_PROTO)
def generate_sequences(self, prompts: DataProto):
    # 1. 将 prompt 数据移到 GPU
    prompts = prompts.to('cuda')
    
    # 2. 使用 sharding manager 同步参数
    # 这里使用的是 MegatronVLLMShardingManager，而非 FSDP 版本
    with self.rollout_sharding_manager:
        # 3. 在上下文管理器内部，会触发参数从 Megatron 模型到 vLLM 引擎的同步
        
        # 4. 预处理数据（例如，根据 TP 维度切分数据）
        prompts = self.rollout_sharding_manager.preprocess_data(prompts)
        
        # 5. ⭐ 核心调用：vLLM 引擎开始生成
        output = self.rollout.generate_sequences(prompts=prompts)
        
        # 6. 后处理数据（例如，聚合来自不同 TP rank 的结果）
        output = self.rollout_sharding_manager.postprocess_data(output)
        
    return output

##### ⚙️ 关键组件：MegatronVLLMShardingManager

这个类是 megatron_workers 的灵魂，负责管理 Megatron 训练引擎和 vLLM 推理引擎之间的参数“搬运”和“重塑”。
- 作用：在 generate_sequences 被调用时，它负责将分布式存储在多个 GPU 上的 Megatron 模型参数，高效地同步到 vLLM 引擎中。生成完毕后，再负责清理 vLLM 中的临时参数，释放显存。
- 工作原理：
 1. 进入上下文 (__enter__)：触发 Megatron 模型将所有参数 gather 起来，并加载到 vLLM 引擎所占用的 GPU 上。
 2. 生成序列：vLLM 引擎使用同步过来的参数进行高效的文本生成。
 3. 退出上下文 (__exit__)：vLLM 引擎释放不再需要的参数，将显存还给训练进程。

### 推理后端

 为了实现高效的文本生成（Rollout），verl.workers 通过适配器（Adapter）模式集成了高性能推理引擎：
 
 | 推理后端 | 集成方式 | 优势 |
| :--- | :--- | :--- |
| vLLM | `vllm_rollout` / `VllmReplica` | 提供高吞吐量的连续批处理生成能力。 |
| SGLang | `sglang_rollout` / `SglangReplica` | 在结构化输出和长上下文推理场景下性能更优。 |
| Hugging Face | `HFRollout` | 方便用于调试或小模型验证。 |
